In [ ]:
import os
import streamlit as st
from openai import OpenAI
import requests
from PIL import Image
from io import BytesIO
import datetime

st.set_page_config(page_title="Photo Generator", page_icon="🖼️")

# App title and description
st.title("Photo Generator")
st.markdown("Generate and save custom images using DALL-E 3")

# Initialize session state variables if they don't exist
if "generated_images" not in st.session_state:
    st.session_state.generated_images = []
if "api_key_set" not in st.session_state:
    st.session_state.api_key_set = False

# Function to create an image using DALL-E
def create_image(client, prompt):
    try:
        response = client.images.generate(
            model="dall-e-3",
            prompt=prompt,
            n=1,
            quality="hd",
            style="vivid",
            size="1024x1024"
        )
        return response.data[0].url
    except Exception as e:
        st.error(f"Error generating image: {str(e)}")
        return None

# Function to save an image
def save_image(image_url):
    try:
        # Download the image
        response = requests.get(image_url)
        response.raise_for_status()
        
        # Open the image using Pillow
        img = Image.open(BytesIO(response.content))
        
        # Convert image to RGB if necessary
        if img.mode != "RGB":
            img = img.convert("RGB")
            
        # Create timestamp for filename
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"generated_image_{timestamp}.jpg"
        
        # Use a more generic folder path or let user select
        folder_path = "generated_images"
        
        # Create folder if it doesn't exist
        if not os.path.exists(folder_path):
            os.makedirs(folder_path)
            
        # Full file path
        file_path = os.path.join(folder_path, filename)
        
        # Save the image
        img.save(file_path, format="JPEG")
        return file_path
    except Exception as e:
        st.error(f"Error saving image: {str(e)}")
        return None

# API key input (with a better UX)
with st.sidebar:
    st.header("API Settings")
    api_key = st.text_input("Enter OpenAI API Key", type="password", help="Your API key will not be stored permanently")
    
    # Set the API key when the user clicks this button
    if st.button("Set API Key"):
        if api_key:
            os.environ["OPENAI_API_KEY"] = api_key
            st.session_state.api_key_set = True
            st.success("API key set successfully!")
        else:
            st.error("Please enter an API key")

# Main form for image generation
with st.form(key="image_generation_form"):
    col1, col2 = st.columns(2)
    
    with col1:
        subject_input = st.text_input("What would you like a photo of?", placeholder="e.g., golden retriever puppy")
    
    with col2:
        bg_color = st.text_input("What color is the background?", placeholder="e.g., light blue")
    
    subject_type = st.selectbox(
        "What is the subject of the photo?",
        ("Animal", "Person", "Object"),
        index=None,
        placeholder="Select a subject type..."
    )
    
    num_images = st.slider("Number of images to generate", min_value=1, max_value=4, value=1)
    
    submit_button = st.form_submit_button("Generate Images")

# Handle form submission
if submit_button:
    if not st.session_state.api_key_set:
        st.error("Please set your API key first")
    elif not subject_input or not bg_color or not subject_type:
        st.warning("Please fill in all fields")
    else:
        # Show a spinner while generating
        with st.spinner("Generating your images..."):
            try:
                # Construct the prompt
                image_prompt = (
                    f"High resolution photograph of a {subject_input}. Plain {bg_color} background. "
                    f"Show entire {subject_type}. The {subject_type} is the singular subject in the photograph. "
                    "Ensure photo is appropriate for children."
                )
                
                # Initialize OpenAI client
                client = OpenAI()
                
                # Clear previous images
                st.session_state.generated_images = []
                
                # Generate the requested number of images
                for i in range(num_images):
                    img_url = create_image(client, image_prompt)
                    if img_url:
                        st.session_state.generated_images.append(img_url)
            
            except Exception as e:
                st.error(f"An error occurred: {str(e)}")

# Display generated images and save buttons
if st.session_state.generated_images:
    st.header("Your Generated Images")
    
    # Calculate number of columns based on image count (1-4)
    num_cols = min(len(st.session_state.generated_images), 2)  # Max 2 columns for better visibility
    
    # Create columns for images
    cols = st.columns(num_cols)
    
    # Display each image with its own save button
    for i, img_url in enumerate(st.session_state.generated_images):
        col_idx = i % num_cols
        with cols[col_idx]:
            st.image(img_url, use_column_width=True)
            if st.button(f"Save Image {i+1}", key=f"save_{i}"):
                saved_path = save_image(img_url)
                if saved_path:
                    st.success(f"Image saved to {saved_path}")

# Instructions at the bottom
with st.expander("How to use this app"):
    st.markdown("""
    1. Enter your OpenAI API key in the sidebar and click "Set API Key"
    2. Fill in what you want in your photo, the background color, and select the subject type
    3. Choose how many images to generate (1-4)
    4. Click "Generate Images" to create your custom images
    5. Use the "Save Image" buttons to save any images you like
    """)

2025-04-12 17:23:06.817 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-12 17:23:06.818 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-12 17:23:06.818 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-12 17:23:06.818 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-12 17:23:06.818 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-12 17:23:06.819 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-12 17:23:06.819 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-12 17:23:06.819 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar